# Stock NeurIPS2018 Part 1. Data
This series is a reproduction of paper *the process in the paper Practical Deep Reinforcement Learning Approach for Stock Trading*. 

This is the first part of the NeurIPS2018 series, introducing how to use FinRL to fetch and process data that we need for ML/RL trading.

Other demos can be found at the repo of [FinRL-Tutorials]((https://github.com/AI4Finance-Foundation/FinRL-Tutorials)).

# Part 1. Install Packages

In [2]:
import pandas as pd
import yfinance as yf

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl import config_tickers
from finrl.config import INDICATORS
from finrl.config import *
import itertools

# Part 2. Fetch data

[yfinance](https://github.com/ranaroussi/yfinance) is an open-source library that provides APIs fetching historical data form Yahoo Finance. In FinRL, we have a class called [YahooDownloader](https://github.com/AI4Finance-Foundation/FinRL/blob/master/finrl/meta/preprocessor/yahoodownloader.py) that use yfinance to fetch data from Yahoo Finance.

**OHLCV**: Data downloaded are in the form of OHLCV, corresponding to **open, high, low, close, volume,** respectively. OHLCV is important because they contain most of numerical information of a stock in time series. From OHLCV, traders can get further judgement and prediction like the momentum, people's interest, market trends, etc.

## Data for a single ticker

Here we provide two ways to fetch data with single ticker, let's take Apple Inc. (AAPL) as an example.

### Using yfinance

In [3]:
TRAIN_START_DATE = '2020-01-01'
TRADE_END_DATE = '2020-01-31'
aapl_df_yf = yf.download(tickers = "aapl", start=TRAIN_START_DATE, end=TRADE_END_DATE)

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed


In [4]:
aapl_df_yf.head()

Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2020-01-02,72.538513,72.598892,71.292304,71.545890,135480400
2020-01-03,71.833290,72.594055,71.608685,71.765667,146322800
2020-01-06,72.405678,72.444321,70.703012,70.954188,118387200
2020-01-07,72.065163,72.671356,71.845385,72.415353,108872000
2020-01-08,73.224403,73.526295,71.768079,71.768079,132079200


### Using FinRL

In FinRL's YahooDownloader, we modified the data frame to the form that convenient for further data processing process. We use adjusted close price instead of close price, and add a column representing the day of a week (0-4 corresponding to Monday-Friday).

In [5]:
aapl_df_finrl = YahooDownloader(start_date = TRAIN_START_DATE,
                                end_date = TRAIN_END_DATE,
                                ticker_list = ['aapl']).fetch_data()

YF deprecation warning: set proxy via new config function: yf.set_config(proxy=proxy)


[*********************100%***********************]  1 of 1 completed

Shape of DataFrame:  (146, 8)


In [6]:
aapl_df_finrl.head()

Price,date,close,high,low,open,volume,tic,day
0,2020-01-02,72.538513,72.598892,71.292304,71.545890,135480400,aapl,3
1,2020-01-03,71.833298,72.594063,71.608692,71.765674,146322800,aapl,4
2,2020-01-06,72.405670,72.444313,70.703005,70.954181,118387200,aapl,0
3,2020-01-07,72.065140,72.671333,71.845362,72.415330,108872000,aapl,1
4,2020-01-08,73.224426,73.526318,71.768101,71.768101,132079200,aapl,2


## Data for the chosen tickers

In [7]:
from base_rl_trading.constants import DOW_TICKERS


config_tickers.DOW_30_TICKER = DOW_TICKERS

In [8]:
TRAIN_START_DATE = '2009-01-01'
TRAIN_END_DATE = '2020-07-01'
TRADE_START_DATE = '2020-07-01'
TRADE_END_DATE = '2021-10-29'

In [9]:
df_raw = YahooDownloader(start_date = TRAIN_START_DATE,
                     end_date = TRADE_END_DATE,
                     ticker_list = config_tickers.DOW_30_TICKER).fetch_data()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

Shape of DataFrame:  (96870, 8)


In [10]:
df_raw.head()

Price,date,close,high,low,open,volume,tic,day
0,2009-01-02,2.724325,2.733032,2.556513,2.578127,746015200,AAPL,4
1,2009-01-02,40.463188,40.524922,39.612630,40.188814,6547900,AMGN,4
2,2009-01-02,2.718000,2.726500,2.553500,2.567500,145928000,AMZN,4
3,2009-01-02,14.854053,15.000058,14.139398,14.270034,10955700,AXP,4
4,2009-01-02,33.941093,34.173619,32.088396,32.103398,7010200,BA,4


# Part 3: Preprocess Data
We need to check for missing data and do feature engineering to convert the data point into a state.
* **Adding technical indicators**. In practical trading, various information needs to be taken into account, such as historical prices, current holding shares, technical indicators, etc. Here, we demonstrate two trend-following technical indicators: MACD and RSI.
* **Adding turbulence index**. Risk-aversion reflects whether an investor prefers to protect the capital. It also influences one's trading strategy when facing different market volatility level. To control the risk in a worst-case scenario, such as financial crisis of 2007–2008, FinRL employs the turbulence index that measures extreme fluctuation of asset price.

Hear let's take **MACD** as an example. Moving average convergence/divergence (MACD) is one of the most commonly used indicator showing bull and bear market. Its calculation is based on EMA (Exponential Moving Average indicator, measuring trend direction over a period of time.)

In [11]:
fe = FeatureEngineer(use_technical_indicator=True,
                     tech_indicator_list = INDICATORS,
                     use_vix=True,
                     use_turbulence=True,
                     user_defined_feature = False)

processed = fe.preprocess_data(df_raw)

Successfully added technical indicators


[*********************100%***********************]  1 of 1 completed


Shape of DataFrame:  (3228, 8)
Successfully added vix
Successfully added turbulence index


In [14]:
list_ticker = processed["tic"].unique().tolist()
list_date = list(pd.date_range(processed['date'].min(),processed['date'].max()).astype(str))
combination = list(itertools.product(list_date,list_ticker))

processed_full = pd.DataFrame(combination,columns=["date","tic"]).merge(processed,on=["date","tic"],how="left")
processed_full = processed_full[processed_full['date'].isin(processed['date'])]
processed_full = processed_full.sort_values(['date','tic'])

processed_full = processed_full.fillna(0)

In [15]:
processed_full.head()

,date,tic,close,high,low,open,volume,day,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,vix,turbulence
0,2009-01-02,AAPL,2.724325,2.733032,2.556513,2.578127,746015200.0,4.0,0.0,2.944418,2.61921,100.0,66.666667,100.0,2.724325,2.724325,39.189999,0.0
1,2009-01-02,AMGN,40.463188,40.524922,39.612630,40.188814,6547900.0,4.0,0.0,2.944418,2.61921,100.0,66.666667,100.0,40.463188,40.463188,39.189999,0.0
2,2009-01-02,AMZN,2.718000,2.726500,2.553500,2.567500,145928000.0,4.0,0.0,2.944418,2.61921,100.0,66.666667,100.0,2.718000,2.718000,39.189999,0.0
3,2009-01-02,AXP,14.854053,15.000058,14.139398,14.270034,10955700.0,4.0,0.0,2.944418,2.61921,100.0,66.666667,100.0,14.854053,14.854053,39.189999,0.0
4,2009-01-02,BA,33.941093,34.173619,32.088396,32.103398,7010200.0,4.0,0.0,2.944418,2.61921,100.0,66.666667,100.0,33.941093,33.941093,39.189999,0.0


# Part 4: Save the Data

### Split the data for training and trading

In [16]:
train = data_split(processed_full, TRAIN_START_DATE,TRAIN_END_DATE)
trade = data_split(processed_full, TRADE_START_DATE,TRADE_END_DATE)
print(len(train))
print(len(trade))

86790
10050


In [17]:
train

,date,tic,close,high,low,open,volume,day,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,vix,turbulence
0,2009-01-02,AAPL,2.724325,2.733032,2.556513,2.578127,746015200.0,4.0,0.000000,2.944418,2.619210,100.000000,66.666667,100.000000,2.724325,2.724325,39.189999,0.000000
0,2009-01-02,AMGN,40.463188,40.524922,39.612630,40.188814,6547900.0,4.0,0.000000,2.944418,2.619210,100.000000,66.666667,100.000000,40.463188,40.463188,39.189999,0.000000
0,2009-01-02,AMZN,2.718000,2.726500,2.553500,2.567500,145928000.0,4.0,0.000000,2.944418,2.619210,100.000000,66.666667,100.000000,2.718000,2.718000,39.189999,0.000000
0,2009-01-02,AXP,14.854053,15.000058,14.139398,14.270034,10955700.0,4.0,0.000000,2.944418,2.619210,100.000000,66.666667,100.000000,14.854053,14.854053,39.189999,0.000000
0,2009-01-02,BA,33.941093,34.173619,32.088396,32.103398,7010200.0,4.0,0.000000,2.944418,2.619210,100.000000,66.666667,100.000000,33.941093,33.941093,39.189999,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2892,2020-06-30,TRV,102.140366,102.534414,100.393990,100.725355,1464500.0,1.0,1.686138,112.348422,96.156786,52.925509,20.975724,1.225613,100.134007,94.375529,30.430000,14.149451
2892,2020-06-30,UNH,271.594116,272.975337,264.881374,265.719318,2932900.0,1.0,-0.018382,286.835069,255.997872,52.413049,-20.026428,0.598575,271.824292,265.199755,30.430000,14.149451
2892,2020-06-30,V,186.097626,186.656392,183.197834,184.479141,9040100.0,1.0,1.023272,193.916018,180.540326,53.021027,-51.428592,2.103749,186.827246,177.258453,30.430000,14.149451
2892,2020-06-30,VZ,39.978672,40.094699,39.420290,39.826384,17414800.0,1.0,-0.346888,42.789333,38.671292,48.097030,-50.671464,8.321391,40.482909,40.842055,30.430000,14.149451


### Save data to csv file

For Colab users, you can open the virtual directory in colab and manually download the files.

For users running on your local environment, the csv files should be at the same directory of this notebook.

In [17]:
train.to_csv('train_data.csv')
trade.to_csv('trade_data.csv')